# YouTube → Dailymotion (merged)

Downloads a YouTube video (or playlist) with `yt-dlp`, splits it under Dailymotion's 2-hour limit, uploads the parts, and posts the resulting URLs back to the ManhuaFlix backend.

**How to use with the backend**

1. Put Dailymotion API credentials in Drive as `/content/drive/MyDrive/dailymotion_credentials.json` (see Step 3).
2. Optional: export YouTube cookies to Drive if you hit "Sign in to confirm you're not a bot".
3. Set `API_BASE` and `COLAB_JOB_SECRET` in Step 2.
4. Runtime → **Run all**. Leave this tab open.
5. From the backend, `POST /api/colab-jobs` with a YouTube URL. Poll `GET /api/colab-jobs/:id` until `status` is `done` — `resultUrls` will contain the Dailymotion links.

**Manual run:** set `JOB_SOURCE` to `form`, fill in the YouTube URL, then Run all.

This notebook needs a CPU runtime only.


In [ ]:
# @title Step 1: Setup
# @markdown Installs tools, mounts Drive, and defines the pipeline helpers.

import json
import os
import re
import shutil
import subprocess
import sys
import time

print("📦 Installing yt-dlp, requests, tqdm, requests-toolbelt, ipywidgets, gdown...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U",
     "yt-dlp[default]", "requests", "tqdm", "requests-toolbelt", "ipywidgets", "gdown"],
    check=True,
    capture_output=True,
)

if subprocess.run(["which", "deno"], capture_output=True).returncode != 0:
    print("📦 Installing Deno (needed by yt-dlp for YouTube JS challenges)...")
    subprocess.run(
        ["bash", "-c", "curl -fsSL https://deno.land/install.sh | sh -s -- -y"],
        check=True,
        capture_output=True,
    )
deno_bin = os.path.expanduser("~/.deno/bin")
if deno_bin not in os.environ.get("PATH", ""):
    os.environ["PATH"] = deno_bin + os.pathsep + os.environ.get("PATH", "")

from google.colab import drive
print("📂 Mounting Google Drive...")
drive.mount("/content/drive")

import requests
from tqdm.notebook import tqdm
from yt_dlp import YoutubeDL

QUALITY_FORMATS = {
    "Best Available": "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best",
    "4K": "bestvideo[height<=2160]+bestaudio/best[height<=2160]/best",
    "1080p": "bestvideo[height<=1080]+bestaudio/best[height<=1080]/best",
    "720p": "bestvideo[height<=720]+bestaudio/best[height<=720]/best",
    "480p": "bestvideo[height<=480]+bestaudio/best[height<=480]/best",
    "360p": "bestvideo[height<=360]+bestaudio/best[height<=360]/best",
}

DEFAULT_DRIVE_FOLDER = "/content/drive/MyDrive/YouTube_Dailymotion_Uploads"
DEFAULT_COOKIES_PATH = "/content/drive/MyDrive/combined_cookies.txt"
DEFAULT_CREDENTIALS_PATH = "/content/drive/MyDrive/dailymotion_credentials.json"


def api_headers(secret: str) -> dict:
    return {"x-colab-secret": secret, "Content-Type": "application/json"}


def claim_job(api_base: str, secret: str) -> dict | None:
    res = requests.get(f"{api_base.rstrip('/')}/colab-jobs/pending", headers=api_headers(secret), timeout=30)
    res.raise_for_status()
    return res.json().get("job")


def complete_job(api_base: str, secret: str, job_id: int, urls: list[dict]) -> dict:
    res = requests.post(
        f"{api_base.rstrip('/')}/colab-jobs/{job_id}/complete",
        headers=api_headers(secret),
        json={"urls": urls},
        timeout=30,
    )
    res.raise_for_status()
    return res.json()


def fail_job(api_base: str, secret: str, job_id: int, error_message: str) -> None:
    try:
        requests.post(
            f"{api_base.rstrip('/')}/colab-jobs/{job_id}/fail",
            headers=api_headers(secret),
            json={"errorMessage": error_message[:4000]},
            timeout=30,
        ).raise_for_status()
    except Exception as err:
        print(f"⚠️ Could not report failure to backend: {err}")


def prefer_mp4(path: str) -> str:
    base, _ext = os.path.splitext(path)
    if os.path.exists(base + ".mp4"):
        return base + ".mp4"
    return path


def download_youtube(job: dict) -> list[str]:
    youtube_url = (job.get("youtubeUrl") or "").strip()
    if not youtube_url:
        raise ValueError("Missing youtubeUrl")

    save_to = (job.get("driveOutputFolder") or DEFAULT_DRIVE_FOLDER).strip()
    os.makedirs(save_to, exist_ok=True)

    quality = job.get("quality") or "1080p"
    if quality not in QUALITY_FORMATS:
        raise ValueError(f"Unknown quality: {quality}")

    is_playlist = bool(job.get("isPlaylist"))
    playlist_range = (job.get("playlistRange") or "").strip()
    cookie_file = (job.get("cookieFilePath") or "").strip() or DEFAULT_COOKIES_PATH
    custom_title = (job.get("customTitle") or "").strip()

    state = {"bar": None}

    def progress_hook(d):
        if d["status"] == "downloading":
            total = d.get("total_bytes") or d.get("total_bytes_estimate")
            downloaded = d.get("downloaded_bytes", 0)
            if state["bar"] is None and total:
                state["bar"] = tqdm(total=total, unit="B", unit_scale=True, unit_divisor=1024, desc="⬇️ Downloading")
            if state["bar"] is not None:
                state["bar"].n = downloaded
                postfix = {}
                speed = d.get("speed")
                eta = d.get("eta")
                if speed:
                    postfix["speed"] = f"{speed/1024/1024:.2f} MB/s"
                if eta is not None:
                    mins, secs = divmod(int(eta), 60)
                    postfix["ETA"] = f"{mins}m {secs}s"
                state["bar"].set_postfix(postfix)
                state["bar"].refresh()
        elif d["status"] == "finished":
            if state["bar"] is not None:
                if state["bar"].total:
                    state["bar"].n = state["bar"].total
                    state["bar"].refresh()
                state["bar"].close()
                state["bar"] = None
            print("🔄 Merging audio/video (if needed)...")

    ydl_opts = {
        "format": QUALITY_FORMATS[quality],
        "merge_output_format": "mp4",
        "outtmpl": os.path.join(save_to, "%(title)s.%(ext)s"),
        "noplaylist": not is_playlist,
        "progress_hooks": [progress_hook],
        "quiet": True,
        "no_warnings": True,
        "retries": 10,
        "fragment_retries": 10,
        "continuedl": True,
        "retry_sleep_functions": {"http": lambda n: 5 * n, "fragment": lambda n: 3 * n},
    }
    if is_playlist and playlist_range:
        ydl_opts["playlist_items"] = playlist_range
    if cookie_file and os.path.exists(cookie_file):
        ydl_opts["cookiefile"] = cookie_file
        print(f"🍪 Using cookies: {cookie_file}")
    elif job.get("cookieFilePath"):
        print(f"Warning: cookies file not found at {cookie_file}. Continuing without cookies.")

    print(f"⬇️ Downloading in {quality}...\n")
    with YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(youtube_url, download=True)
        paths = []
        entries = info.get("entries") if info.get("_type") == "playlist" else [info]
        for entry in entries or []:
            if not entry:
                continue
            path = prefer_mp4(ydl.prepare_filename(entry))
            paths.append(path)

    if not paths:
        raise RuntimeError("Download finished but no video files were produced.")

    if custom_title and len(paths) == 1:
        new_path = os.path.join(os.path.dirname(paths[0]), custom_title + ".mp4")
        if os.path.abspath(new_path) != os.path.abspath(paths[0]) and os.path.exists(paths[0]):
            os.rename(paths[0], new_path)
            paths[0] = new_path

    for path in paths:
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"✅ Downloaded: {os.path.basename(path)} ({size_mb:.1f} MB)")
        print(f"   Saved to: {path}")
    return paths


def get_duration(filepath: str) -> float:
    cmd = ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_format", filepath]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=True)
    return float(json.loads(result.stdout)["format"]["duration"])


def split_video_by_duration(input_path: str, output_dir: str, chunk_seconds: int, title: str, start_number: int) -> list[str]:
    total_duration = get_duration(input_path)
    os.makedirs(output_dir, exist_ok=True)

    if total_duration <= chunk_seconds:
        print("ℹ️ Video is already within the length limit — no split needed.")
        single_path = os.path.join(output_dir, f"{title} - Part {start_number:03d}.mp4")
        if os.path.abspath(single_path) != os.path.abspath(input_path):
            shutil.copy(input_path, single_path)
        return [single_path]

    num_parts = -(-int(total_duration) // chunk_seconds)
    h, m = chunk_seconds // 3600, (chunk_seconds % 3600) // 60
    print(f"🎬 Video is {total_duration/3600:.2f}h → {num_parts} part(s) of up to {h}h {m}m, starting at Part {start_number}\n")

    tmp_pattern = os.path.join(output_dir, "_raw_part_%03d.mp4")
    split_cmd = [
        "ffmpeg", "-y", "-i", input_path,
        "-c", "copy", "-map", "0",
        "-segment_time", str(chunk_seconds),
        "-f", "segment", "-reset_timestamps", "1",
        "-progress", "pipe:1", "-nostats",
        tmp_pattern,
    ]
    process = subprocess.Popen(split_cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True, bufsize=1)
    pbar = tqdm(total=round(total_duration), unit="s", desc="✂️ Splitting")
    last_sec = 0
    for line in process.stdout:
        line = line.strip()
        if line.startswith("out_time_ms="):
            try:
                current_sec = int(line.split("=")[1]) / 1_000_000
                delta = current_sec - last_sec
                if delta > 0:
                    pbar.update(min(delta, max(0, pbar.total - pbar.n)))
                    last_sec = current_sec
            except ValueError:
                pass
        elif line == "progress=end":
            pbar.n = pbar.total
            pbar.refresh()
    process.wait()
    pbar.close()
    if process.returncode != 0:
        raise RuntimeError("ffmpeg failed while splitting the video.")

    raw_parts = sorted(f for f in os.listdir(output_dir) if f.startswith("_raw_part_") and f.endswith(".mp4"))
    final_parts = []
    for i, raw_name in enumerate(raw_parts):
        part_num = start_number + i
        final_path = os.path.join(output_dir, f"{title} - Part {part_num:03d}.mp4")
        os.rename(os.path.join(output_dir, raw_name), final_path)
        final_parts.append(final_path)
        print(f"   • {os.path.basename(final_path)}")
    return final_parts


class DailyLimitReached(Exception):
    pass


DAILY_LIMIT_HINTS = ["limit", "quota", "too many", "daily", "rate limit"]
PART_PATTERN = re.compile(r"^(.*) - Part (\d+)$")


def is_daily_limit_error(response) -> bool:
    text = (response.text or "").lower()
    return response.status_code in (403, 429) and any(hint in text for hint in DAILY_LIMIT_HINTS)


def dailymotion_login(credentials: dict) -> str:
    print("🔐 Authenticating with Dailymotion...")
    response = requests.post(
        "https://api.dailymotion.com/oauth/token",
        data={
            "grant_type": "password",
            "client_id": credentials["client_id"],
            "client_secret": credentials["client_secret"],
            "username": credentials["username"],
            "password": credentials["password"],
            "scope": "manage_videos userinfo",
        },
        timeout=30,
    )
    if response.status_code != 200:
        raise RuntimeError(f"Dailymotion auth failed: {response.status_code} {response.text}")
    print("✅ Logged in to Dailymotion.")
    return response.json()["access_token"]


def load_dailymotion_credentials(path: str) -> dict:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing {path}. Create it with keys: client_id, client_secret, username, password"
        )
    with open(path, "r", encoding="utf-8") as handle:
        creds = json.load(handle)
    required = ("client_id", "client_secret", "username", "password")
    missing = [key for key in required if not creds.get(key)]
    if missing:
        raise ValueError(f"Dailymotion credentials missing: {', '.join(missing)}")
    return creds


def upload_to_dailymotion(file_path: str, title: str, part_number: int | None, token: str, channel: str, tags: str) -> tuple[str, str, str]:
    from requests_toolbelt.multipart.encoder import MultipartEncoder, MultipartEncoderMonitor

    headers = {"Authorization": f"Bearer {token}"}
    res = requests.get("https://api.dailymotion.com/file/upload", headers=headers, timeout=30)
    if is_daily_limit_error(res):
        raise DailyLimitReached(res.text)
    res.raise_for_status()
    upload_url = res.json()["url"]

    file_size = os.path.getsize(file_path)
    pbar = tqdm(total=file_size, unit="B", unit_scale=True, unit_divisor=1024, desc=f"⬆️ {os.path.basename(file_path)}")
    start_time = time.time()

    def callback(monitor):
        pbar.n = monitor.bytes_read
        elapsed = time.time() - start_time
        if elapsed > 0 and monitor.bytes_read > 0:
            speed = monitor.bytes_read / elapsed
            remaining = monitor.bytes_total - monitor.bytes_read
            eta_sec = remaining / speed if speed > 0 else 0
            mins, secs = divmod(int(eta_sec), 60)
            pbar.set_postfix({"speed": f"{speed/1024/1024:.2f} MB/s", "ETA": f"{mins}m {secs}s"})
        pbar.refresh()

    with open(file_path, "rb") as handle:
        encoder = MultipartEncoder(fields={"file": (os.path.basename(file_path), handle, "video/mp4")})
        monitor = MultipartEncoderMonitor(encoder, callback)
        res = requests.post(upload_url, data=monitor, headers={"Content-Type": monitor.content_type}, timeout=None)
        if is_daily_limit_error(res):
            pbar.close()
            raise DailyLimitReached(res.text)
        res.raise_for_status()
        video_url = res.json()["url"]

    pbar.n = file_size
    pbar.refresh()
    pbar.close()

    final_title = f"{title} - Part {part_number}" if part_number else title
    print(f"   📝 Publishing '{final_title}'...")
    res = requests.post(
        "https://api.dailymotion.com/me/videos",
        headers=headers,
        data={"url": video_url, "title": final_title, "channel": channel, "tags": tags, "published": "true"},
        timeout=60,
    )
    if is_daily_limit_error(res):
        raise DailyLimitReached(res.text)
    res.raise_for_status()
    video_id = res.json()["id"]
    public_url = f"https://www.dailymotion.com/video/{video_id}"
    print(f"   ✅ {public_url}")
    return video_id, final_title, public_url


def title_and_part_for(filepath: str, title_override: str) -> tuple[str, int | None]:
    name = os.path.splitext(os.path.basename(filepath))[0]
    match = PART_PATTERN.match(name)
    if match:
        base_title, part_num = match.group(1), int(match.group(2))
    else:
        base_title, part_num = name, None
    if title_override:
        base_title = title_override
    return base_title, part_num


def upload_parts(part_paths: list[str], token: str, job: dict) -> list[dict]:
    channel = job.get("dailymotionChannel") or "news"
    tags = job.get("dailymotionTags") or "youtube,upload"
    title_override = (job.get("customTitle") or "").strip()
    uploaded = []
    remaining = list(part_paths)

    for file_path in part_paths:
        title, part_num = title_and_part_for(file_path, title_override)
        print(f"\n--- {os.path.basename(file_path)} ---")
        try:
            video_id, vid_title, public_url = upload_to_dailymotion(file_path, title, part_num, token, channel, tags)
            uploaded.append({"title": vid_title, "videoId": video_id, "url": public_url})
            remaining.remove(file_path)
        except DailyLimitReached:
            leftover = ", ".join(os.path.basename(path) for path in remaining)
            done = ", ".join(item["url"] for item in uploaded) or "none"
            raise DailyLimitReached(
                f"Dailymotion daily upload limit reached after {len(uploaded)} part(s). "
                f"Uploaded: {done}. Remaining: {leftover}"
            ) from None

    return uploaded


def run_pipeline(job: dict, access_token: str) -> list[dict]:
    video_paths = download_youtube(job)
    chunk_seconds = int(job.get("splitHours") or 1) * 3600 + int(job.get("splitMinutes") or 55) * 60
    start_part = int(job.get("startPartNumber") or 1)
    all_parts = []
    for video_path in video_paths:
        title = os.path.splitext(os.path.basename(video_path))[0]
        parts_dir = os.path.join(os.path.dirname(video_path), f"{title}_parts")
        parts = split_video_by_duration(video_path, parts_dir, chunk_seconds, title, start_part)
        all_parts.extend(parts)
        start_part += len(parts)
    return upload_parts(all_parts, access_token, job)


print("\n✅ Setup complete. Drive is at /content/drive")


In [ ]:
# @title Step 2: Job source
# @markdown `api` = pull a job from the backend. `form` = use the fields below.

JOB_SOURCE = "api"  # @param ["api", "form"]
API_BASE = "https://your-backend.vercel.app/api"  # @param {type:"string"}
COLAB_JOB_SECRET = ""  # @param {type:"string"}
LOOP_JOBS = True  # @param {type:"boolean"}
POLL_SECONDS = 30  # @param {type:"integer"}

# @markdown ---
# @markdown **Form mode only**
YOUTUBE_URL = ""  # @param {type:"string"}
CUSTOM_TITLE = ""  # @param {type:"string"}
VIDEO_QUALITY = "1080p"  # @param ["Best Available", "4K", "1080p", "720p", "480p", "360p"]
IS_PLAYLIST = False  # @param {type:"boolean"}
PLAYLIST_RANGE = ""  # @param {type:"string"}
COOKIE_FILE_PATH = "/content/drive/MyDrive/combined_cookies.txt"  # @param {type:"string"}
DRIVE_OUTPUT_FOLDER = "/content/drive/MyDrive/YouTube_Dailymotion_Uploads"  # @param {type:"string"}
SPLIT_HOURS = 1  # @param {type:"integer"}
SPLIT_MINUTES = 55  # @param {type:"integer"}
START_PART_NUMBER = 1  # @param {type:"integer"}
DAILYMOTION_CHANNEL = "news"  # @param ["news", "music", "sport", "tech", "creation", "auto"]
DAILYMOTION_TAGS = "youtube,upload"  # @param {type:"string"}

def job_from_form() -> dict:
    if not YOUTUBE_URL.strip():
        raise ValueError("Paste a YouTube URL in the form, or switch JOB_SOURCE to api.")
    return {
        "id": None,
        "youtubeUrl": YOUTUBE_URL.strip(),
        "customTitle": CUSTOM_TITLE.strip() or None,
        "quality": VIDEO_QUALITY,
        "isPlaylist": IS_PLAYLIST,
        "playlistRange": PLAYLIST_RANGE.strip() or None,
        "cookieFilePath": COOKIE_FILE_PATH.strip() or None,
        "driveOutputFolder": DRIVE_OUTPUT_FOLDER.strip() or DEFAULT_DRIVE_FOLDER,
        "splitHours": SPLIT_HOURS,
        "splitMinutes": SPLIT_MINUTES,
        "startPartNumber": START_PART_NUMBER,
        "dailymotionChannel": DAILYMOTION_CHANNEL,
        "dailymotionTags": DAILYMOTION_TAGS,
    }

print(f"Job source: {JOB_SOURCE}")
if JOB_SOURCE == "api":
    if not API_BASE.strip() or "your-backend" in API_BASE:
        raise ValueError("Set API_BASE to your deployed backend, including /api")
    if not COLAB_JOB_SECRET.strip():
        raise ValueError("Set COLAB_JOB_SECRET to match the backend env var")
    print(f"Backend: {API_BASE.rstrip('/')}")


In [ ]:
# @title Optional: merge YouTube + Google cookie exports
# @markdown Skip this if you already have one Netscape `cookies.txt`. Re-run Step 4 after merging.

import json as _json

youtube_cookies_path = "/content/drive/MyDrive/youtube_cookies.txt"  # @param {type:"string"}
google_cookies_path = "/content/drive/MyDrive/google_cookies.txt"  # @param {type:"string"}
combined_cookies_path = "/content/drive/MyDrive/combined_cookies.txt"  # @param {type:"string"}

def _json_to_netscape(cookies):
    lines = ["# Netscape HTTP Cookie File"]
    for cookie in cookies:
        expiry = int(cookie.get("expirationDate", cookie.get("expires", 0)))
        if expiry > 253402300000:
            expiry //= 1000
        domain = cookie.get("domain", "")
        host_only = cookie.get("hostOnly", False)
        if not host_only and not domain.startswith("."):
            domain = "." + domain
        flag = "FALSE" if host_only else "TRUE"
        path = cookie.get("path", "/")
        secure = "TRUE" if cookie.get("secure", False) else "FALSE"
        name = cookie.get("name", "")
        value = cookie.get("value", "") or ""
        lines.append(f"{domain}\t{flag}\t{path}\t{secure}\t{expiry}\t{name}\t{value}")
    return "\n".join(lines)

def _read_cookie_file(path):
    if not os.path.exists(path):
        print(f"Warning: not found: {path}")
        return ""
    content = open(path, "r", encoding="utf-8").read()
    try:
        return _json_to_netscape(_json.loads(content))
    except _json.JSONDecodeError:
        return content

combined = [chunk for chunk in (_read_cookie_file(youtube_cookies_path), _read_cookie_file(google_cookies_path)) if chunk]
if combined:
    with open(combined_cookies_path, "w", encoding="utf-8") as handle:
        handle.write("\n".join(combined))
    print(f"Combined cookies written to: {combined_cookies_path}")
else:
    print("Nothing to combine — check the file paths.")


In [ ]:
# @title Step 3: Dailymotion credentials
# @markdown Reads `/content/drive/MyDrive/dailymotion_credentials.json`. Example:
# @markdown
# @markdown `{ "client_id": "", "client_secret": "", "username": "", "password": "" }`

DAILYMOTION_CREDENTIALS_PATH = "/content/drive/MyDrive/dailymotion_credentials.json"  # @param {type:"string"}

DM_CREDENTIALS = load_dailymotion_credentials(DAILYMOTION_CREDENTIALS_PATH)
ACCESS_TOKEN = dailymotion_login(DM_CREDENTIALS)


In [ ]:
# @title Step 4: Run pipeline
# @markdown In `api` mode this claims pending jobs, processes them, and posts Dailymotion URLs back.
# @markdown Leave it running. Free Colab will still disconnect after idle / max session time.

def process_one(job: dict) -> list[dict]:
    job_id = job.get("id")
    print(f"\n{'=' * 60}\n▶ Job {job_id or '(manual)'} — {job.get('youtubeUrl')}\n{'=' * 60}")
    try:
        urls = run_pipeline(job, ACCESS_TOKEN)
    except Exception as err:
        if job_id is not None:
            fail_job(API_BASE, COLAB_JOB_SECRET, job_id, str(err))
        raise
    print("\n🎉 Upload summary:")
    for item in urls:
        print(f"   • {item['title']}: {item['url']}")
    if job_id is not None:
        complete_job(API_BASE, COLAB_JOB_SECRET, job_id, urls)
        print(f"✅ Reported {len(urls)} URL(s) to job #{job_id}")
    return urls


if JOB_SOURCE == "form":
    process_one(job_from_form())
else:
    processed = 0
    while True:
        job = claim_job(API_BASE, COLAB_JOB_SECRET)
        if job is None:
            if processed == 0 or LOOP_JOBS:
                print(f"No pending jobs. Waiting {POLL_SECONDS}s...")
                time.sleep(max(5, int(POLL_SECONDS)))
                continue
            break
        process_one(job)
        processed += 1
        if not LOOP_JOBS:
            break


## Troubleshooting

**Backend returns 401 on pending/complete**
- `COLAB_JOB_SECRET` in this notebook must match the `COLAB_JOB_SECRET` env var on Vercel.

**"Sign in to confirm you're not a bot"**
- Re-export cookies while logged into YouTube, merge them in the optional cell, then re-run Step 4.
- Avoid VPNs / datacenter IPs.

**Only thumbnail/storyboard formats**
- Re-run Step 1 so `yt-dlp[default]` and Deno actually install.

**Dailymotion daily upload limit**
- The job is marked `failed`. Wait ~24 hours and create a new job pointing at the same YouTube URL, or re-run form mode on the leftover parts folder.

**Colab disconnected / Vercel timed out**
- That is expected on free tiers. Queue the job from the backend (`POST /api/colab-jobs`), then open this notebook and Run all. Poll `GET /api/colab-jobs/:id` for `resultUrls` — do not wait on the create request.
